In [1]:
"""
NB04_Encounter_Detector_Gating_and_Segmentation

Purpose
-------
1) Load disturbing-acceleration samples from NB03 artifact.
2) Smooth ||dg|| along true anomaly (Section 5.4).
3) Apply threshold gating (percentile or absolute) to define encounter windows.
4) Merge close segments, discard short ones, handle orbit wrap-around.
5) Optionally apply hysteresis entry/exit thresholds (Section 3.3).
6) Annotate each encounter with geographic footprint (lat/lon at peak dg).
7) Save encounter windows and descriptors to .npz + .json.

Dependencies
------------
NB03 artifact (.npz) in ./artifacts/
Optionally CONFIG from NB00 for gate parameters.

Outputs
-------
artifacts/encounters_<tag>.npz
artifacts/encounters_<tag>.json
"""

from __future__ import annotations

import json
import math
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

In [2]:
# ============================================================
# Cell 1 : Configuration — Gate Parameters from CONFIG
# ============================================================
try:
    _gate = CONFIG["gate"]
    GATE_MODE   = _gate["mode"]                # "percentile" or "absolute"
    PERCENTILE  = _gate["percentile"]           # used when mode=="percentile"
    G_TH_ABS    = _gate["g_th_abs"]             # used when mode=="absolute"
    SMOOTH_WIN  = _gate["smoothing"]["window_points"]
    SMOOTH_ON   = _gate["smoothing"]["enabled"]
    HYST_ON     = _gate["hysteresis"]["enabled"]
    HYST_HIGH   = _gate["hysteresis"]["high_multiplier"]
    HYST_LOW    = _gate["hysteresis"]["low_multiplier"]
    MIN_LEN     = _gate["min_window_points"]
    MERGE_GAP   = _gate["merge_gap_points"]
    _cfg_source = "CONFIG (NB00)"
except NameError:
    GATE_MODE   = "percentile"
    PERCENTILE  = 90.0
    G_TH_ABS    = None
    SMOOTH_WIN  = 31
    SMOOTH_ON   = True
    HYST_ON     = False
    HYST_HIGH   = 1.05
    HYST_LOW    = 0.95
    MIN_LEN     = 20
    MERGE_GAP   = 10
    _cfg_source = "FALLBACK (NB00-consistent)"

WRAP_AROUND = True  # treat orbit as closed (nu wraps 2pi -> 0)

print(f"Gate config source : {_cfg_source}")
print(f"  mode        = {GATE_MODE}")
print(f"  percentile  = {PERCENTILE}")
print(f"  g_th_abs    = {G_TH_ABS}")
print(f"  smoothing   = {SMOOTH_ON}  (window={SMOOTH_WIN})")
print(f"  hysteresis  = {HYST_ON}  (high={HYST_HIGH}, low={HYST_LOW})")
print(f"  min_len     = {MIN_LEN}")
print(f"  merge_gap   = {MERGE_GAP}")
print(f"  wrap_around = {WRAP_AROUND}")

Gate config source : FALLBACK (NB00-consistent)
  mode        = percentile
  percentile  = 90.0
  g_th_abs    = None
  smoothing   = True  (window=31)
  hysteresis  = False  (high=1.05, low=0.95)
  min_len     = 20
  merge_gap   = 10
  wrap_around = True


In [3]:
# ============================================================
# Cell 2 : Locate and Load NB03 Artifact
# ============================================================
ART_DIR = Path("./artifacts")

# Auto-discover most recent NB03 artifact
nb03_files = sorted(ART_DIR.glob("dg_samples_NB03_*_L*.npz"))
if not nb03_files:
    raise FileNotFoundError("No NB03 artifact found in ./artifacts/. Run NB03 first.")

IN_NPZ = nb03_files[-1]  # most recent by filename (timestamp-sorted)

RUN_TAG = datetime.now(timezone.utc).strftime("NB04_%Y%m%dT%H%M%SZ")
OUT_NPZ  = ART_DIR / f"encounters_{RUN_TAG}.npz"
OUT_JSON = ART_DIR / f"encounters_{RUN_TAG}.json"

print(f"Input  : {IN_NPZ}")
print(f"Output : {OUT_NPZ}")
print(f"Output : {OUT_JSON}")

Input  : artifacts\dg_samples_NB03_20260316T033826Z_L200.npz
Output : artifacts\encounters_NB04_20260316T035140Z.npz
Output : artifacts\encounters_NB04_20260316T035140Z.json


In [4]:
# ============================================================
# Cell 3 : Load NB03 Data
# ============================================================
d = np.load(IN_NPZ, allow_pickle=False)

nu      = d["nu"]       # (M,)
t       = d["t"]        # (M,)
AR      = d["AR"]       # (M,)
AT      = d["AT"]       # (M,)
AN      = d["AN"]       # (M,)
r_fixed = d["r_fixed"]  # (M, 3) — for geographic footprint

# Corrected NB03 saves "dg_mag"; old NB03 saved "dg_mag_eci".
if "dg_mag" in d:
    dg_mag = d["dg_mag"]
elif "dg_mag_eci" in d:
    dg_mag = d["dg_mag_eci"]
else:
    # Reconstruct from RTN components
    dg_mag = np.sqrt(AR**2 + AT**2 + AN**2)

M  = nu.size
dt = float(np.median(np.diff(t)))

# Load metadata if present
if "meta_json" in d:
    _nb03_meta = json.loads(str(d["meta_json"]))
    _L_used = _nb03_meta.get("gravity_model", {}).get("L_used", "?")
else:
    _nb03_meta = {}
    _L_used = "?"

print(f"Loaded NB03 artifact: {IN_NPZ.name}")
print(f"  M    = {M}")
print(f"  dt   ~ {dt:.4f} s")
print(f"  L    = {_L_used}")
print(f"  dg magnitude [km/s^2]:  min={dg_mag.min():.6e}  mean={dg_mag.mean():.6e}  max={dg_mag.max():.6e}")

Loaded NB03 artifact: dg_samples_NB03_20260316T033826Z_L200.npz
  M    = 4000
  dt   ~ 1.6952 s
  L    = 200
  dg magnitude [km/s^2]:  min=1.941380e-07  mean=7.142456e-07  max=1.701346e-06


In [5]:
# ============================================================
# Cell 4 : Smoothing and Gating Functions
# ============================================================

def moving_average(x: np.ndarray, w: int) -> np.ndarray:
    """Causal-free (centred) moving average with edge padding."""
    if w <= 1:
        return x.copy()
    w = int(w)
    kernel = np.ones(w, dtype=float) / w
    pad = w // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    return np.convolve(xp, kernel, mode="valid")


def mask_to_segments(mask: np.ndarray):
    """Return list of [start, end] inclusive index pairs for contiguous True runs."""
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return []
    segs = []
    s = prev = idx[0]
    for k in idx[1:]:
        if k == prev + 1:
            prev = k
        else:
            segs.append([int(s), int(prev)])
            s = prev = k
    segs.append([int(s), int(prev)])
    return segs


def merge_close_segments(segs, gap: int):
    """Merge segments separated by <= gap samples."""
    if not segs:
        return []
    segs = sorted(segs, key=lambda ab: ab[0])
    out = [list(segs[0])]
    for a, b in segs[1:]:
        if a <= out[-1][1] + gap + 1:
            out[-1][1] = max(out[-1][1], b)
        else:
            out.append([a, b])
    return out


def filter_min_len(segs, min_len: int):
    """Discard segments shorter than min_len samples."""
    return [[a, b] for a, b in segs if (b - a + 1) >= min_len]


def handle_wraparound(segs, M: int, gap: int):
    """
    If the last and first segments are close through the orbit wrap boundary,
    merge them into a single wraparound segment stored as [last_start, first_end]
    with last_start > first_end (modular convention).
    """
    if len(segs) < 2:
        return segs
    segs = sorted(segs, key=lambda ab: ab[0])
    first = segs[0]
    last  = segs[-1]
    wrap_gap = (M - 1 - last[1]) + first[0]
    if wrap_gap <= gap:
        merged = [last[0], first[1]]  # a > b signals wraparound
        return [merged] + segs[1:-1]
    return segs


def hysteresis_gate(signal: np.ndarray, g_high: float, g_low: float) -> np.ndarray:
    """
    Hysteresis gating (Section 3.3):
    Encounter BEGINS when signal >= g_high,
    TERMINATES when signal <= g_low.
    Returns boolean mask.
    """
    mask = np.zeros(len(signal), dtype=bool)
    inside = False
    for k in range(len(signal)):
        if inside:
            if signal[k] <= g_low:
                inside = False
            else:
                mask[k] = True
        else:
            if signal[k] >= g_high:
                inside = True
                mask[k] = True
    return mask


def get_encounter_indices(a: int, b: int, M: int) -> np.ndarray:
    """Return index array for segment [a, b], handling wraparound (a > b)."""
    if a <= b:
        return np.arange(a, b + 1)
    else:
        return np.concatenate([np.arange(a, M), np.arange(0, b + 1)])


print("Gating functions defined.")

Gating functions defined.


In [6]:
# ============================================================
# Cell 5 : Apply Smoothing + Gating + Segmentation
# ============================================================

# --- Step 1: Smooth the disturbance magnitude ---
metric = dg_mag.copy()
if SMOOTH_ON and SMOOTH_WIN > 1:
    metric_s = moving_average(metric, SMOOTH_WIN)
    print(f"Smoothed with moving average (window={SMOOTH_WIN}).")
else:
    metric_s = metric.copy()
    print("No smoothing applied.")

# --- Step 2: Determine threshold ---
if GATE_MODE == "percentile":
    thr = float(np.percentile(metric_s, PERCENTILE))
    print(f"Threshold (P{PERCENTILE}) = {thr:.6e} km/s^2")
elif GATE_MODE == "absolute":
    if G_TH_ABS is None:
        raise ValueError("GATE_MODE='absolute' but g_th_abs is None.")
    thr = float(G_TH_ABS)
    print(f"Threshold (absolute) = {thr:.6e} km/s^2")
else:
    raise ValueError(f"Unknown GATE_MODE: {GATE_MODE}")

# --- Step 3: Gate ---
if HYST_ON:
    g_high = thr * HYST_HIGH
    g_low  = thr * HYST_LOW
    mask = hysteresis_gate(metric_s, g_high, g_low)
    print(f"Hysteresis: g_high={g_high:.6e}, g_low={g_low:.6e}")
else:
    mask = metric_s >= thr

print(f"Gated fraction = {mask.mean():.4f}  ({mask.sum()} / {M} samples)")

# --- Step 4: Segment, merge, filter ---
segs = mask_to_segments(mask)
print(f"Raw segments: {len(segs)}")

segs = merge_close_segments(segs, MERGE_GAP)
print(f"After merge (gap<={MERGE_GAP}): {len(segs)}")

if WRAP_AROUND:
    segs = handle_wraparound(segs, M, MERGE_GAP)
    print(f"After wraparound check: {len(segs)}")

segs = filter_min_len(segs, MIN_LEN)
print(f"After min_len filter (>={MIN_LEN}): {len(segs)}")

print(f"\n{'='*70}")
print(f"  ENCOUNTER WINDOWS: {len(segs)} detected")
print(f"{'='*70}")
print(f"{'ID':>3s}  {'Start':>5s}  {'End':>5s}  {'Len':>4s}  {'nu_start':>8s}  {'nu_end':>8s}  {'Peak dg':>10s}")
for i, (a, b) in enumerate(segs):
    idx = get_encounter_indices(a, b, M)
    wrap_tag = "*" if a > b else " "
    print(f"{i:3d}{wrap_tag} {a:5d}  {b:5d}  {idx.size:4d}  {nu[idx[0]]:8.4f}  {nu[idx[-1]]:8.4f}  {metric_s[idx].max():10.4e}")

Smoothed with moving average (window=31).
Threshold (P90.0) = 1.103803e-06 km/s^2
Gated fraction = 0.1000  (400 / 4000 samples)
Raw segments: 6
After merge (gap<=10): 6
After wraparound check: 6
After min_len filter (>=20): 4

  ENCOUNTER WINDOWS: 4 detected
 ID  Start    End   Len  nu_start    nu_end     Peak dg
  0   1187   1315   129    1.8645    2.0656  1.6673e-06
  1   1566   1615    50    2.4599    2.5368  1.2669e-06
  2   2073   2122    50    3.2563    3.3332  1.2831e-06
  3   2837   2976   140    4.4563    4.6747  1.3403e-06


In [7]:
# ============================================================
# Cell 6 : Build Encounter Descriptors with Geographic Footprint
# ============================================================
# For each encounter, compute:
#   - Anomaly / time bounds
#   - Peak and mean metric
#   - Geographic footprint: lat/lon of peak-dg sample in Moon-fixed frame
#     (Section 5.4: "latitude / longitude of the sub-satellite point at peak disturbance")

def cartesian_to_lonlat(r_fixed_km: np.ndarray):
    """Convert Moon-fixed Cartesian position to (lon_deg, lat_deg)."""
    x, y, z = float(r_fixed_km[0]), float(r_fixed_km[1]), float(r_fixed_km[2])
    r = math.sqrt(x*x + y*y + z*z)
    lon = math.degrees(math.atan2(y, x))        # east-positive, [-180, 180]
    lat = math.degrees(math.asin(max(-1.0, min(1.0, z / r))))
    return lon, lat


# Build enc_id array (per-sample encounter label, -1 = outside)
enc_id = -np.ones(M, dtype=int)
for i, (a, b) in enumerate(segs):
    idx = get_encounter_indices(a, b, M)
    enc_id[idx] = i

# Build encounter descriptor list
encounters = []
for i, (a, b) in enumerate(segs):
    idx = get_encounter_indices(a, b, M)
    peak_local = np.argmax(metric_s[idx])   # index within the encounter
    peak_global = idx[peak_local]            # index in the full orbit grid

    lon_peak, lat_peak = cartesian_to_lonlat(r_fixed[peak_global])

    encounters.append({
        "id":           int(i),
        "start":        int(a),
        "end":          int(b),
        "length":       int(idx.size),
        "wraparound":   bool(a > b),
        # Time bounds
        "t_start_s":    float(t[idx[0]]),
        "t_end_s":      float(t[idx[-1]]),
        # Anomaly bounds
        "nu_start_rad": float(nu[idx[0]]),
        "nu_end_rad":   float(nu[idx[-1]]),
        # Metric statistics
        "metric_peak":  float(metric_s[idx].max()),
        "metric_mean":  float(metric_s[idx].mean()),
        # Geographic footprint at peak (Section 5.4)
        "peak_idx":     int(peak_global),
        "peak_lon_deg": round(lon_peak, 3),
        "peak_lat_deg": round(lat_peak, 3),
    })

print(f"Built {len(encounters)} encounter descriptors.")
print(f"\n{'ID':>3s}  {'Len':>4s}  {'Peak dg':>10s}  {'Lat':>7s}  {'Lon':>8s}")
for enc in encounters:
    print(f"{enc['id']:3d}  {enc['length']:4d}  {enc['metric_peak']:10.4e}  {enc['peak_lat_deg']:7.2f}  {enc['peak_lon_deg']:8.2f}")

Built 4 encounter descriptors.

 ID   Len     Peak dg      Lat       Lon
  0   129  1.6673e-06   -67.68    179.68
  1    50  1.2669e-06   -36.54    179.59
  2    50  1.2831e-06     8.82    179.46
  3   140  1.3403e-06    85.50    179.24


In [8]:
# ============================================================
# Cell 7 : Save Artifacts
# ============================================================
meta = {
    "source_npz":      str(IN_NPZ),
    "run_tag":         RUN_TAG,
    "metric":          "dg_mag (smoothed) gated by " + GATE_MODE,
    "params": {
        "gate_mode":     GATE_MODE,
        "percentile":    PERCENTILE,
        "g_th_abs":      G_TH_ABS,
        "smooth_on":     SMOOTH_ON,
        "smooth_win":    SMOOTH_WIN,
        "hysteresis_on": HYST_ON,
        "hyst_high":     HYST_HIGH,
        "hyst_low":      HYST_LOW,
        "min_len":       MIN_LEN,
        "merge_gap":     MERGE_GAP,
        "wrap_around":   WRAP_AROUND,
    },
    "threshold":        thr,
    "num_encounters":   len(encounters),
}

np.savez(
    OUT_NPZ,
    # Full orbit arrays
    nu=nu,
    t=t,
    metric=metric,          # raw dg_mag
    metric_s=metric_s,      # smoothed
    threshold=thr,
    mask=mask.astype(np.uint8),
    enc_id=enc_id,
    # Segment boundaries
    segs=np.array(segs, dtype=int) if segs else np.zeros((0, 2), dtype=int),
    # RTN components (pass through for NB05 convenience)
    AR=AR, AT=AT, AN=AN,
    # Metadata
    meta_json=json.dumps(meta),
)

OUT_JSON.write_text(
    json.dumps({"meta": meta, "encounters": encounters}, indent=2),
    encoding="utf-8"
)

print(f"Saved: {OUT_NPZ}")
print(f"Saved: {OUT_JSON}")
print(f"Keys : {sorted(np.load(OUT_NPZ).files)}")
print(f"\n--- NB04 complete ---")

Saved: artifacts\encounters_NB04_20260316T035140Z.npz
Saved: artifacts\encounters_NB04_20260316T035140Z.json
Keys : ['AN', 'AR', 'AT', 'enc_id', 'mask', 'meta_json', 'metric', 'metric_s', 'nu', 'segs', 't', 'threshold']

--- NB04 complete ---
